# Autoencoders and VAEs

Train a small variational autoencoder on local two-dimensional Gaussian data. Inspect reconstruction, latent sampling, and the KL term.

In [ ]:
import torch
from torch import nn

torch.manual_seed(38)
torch.set_num_threads(1)
print('CPU | two-dimensional VAE')

In [ ]:
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(2, 16), nn.ReLU())
        self.mean = nn.Linear(16, 2)
        self.log_variance = nn.Linear(16, 2)
        self.decoder = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))
    def forward(self, x):
        hidden = self.encoder(x)
        mean, log_variance = self.mean(hidden), self.log_variance(hidden)
        latent = mean + torch.exp(0.5 * log_variance) * torch.randn_like(mean)
        return self.decoder(latent), mean, log_variance

data = torch.randn(256, 2) * 0.7 + torch.tensor([1., -1.])
model = VAE()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
for _ in range(120):
    reconstruction, mean, log_variance = model(data)
    reconstruction_loss = nn.functional.mse_loss(reconstruction, data)
    kl = -0.5 * (1 + log_variance - mean.square() - log_variance.exp()).sum(1).mean()
    loss = reconstruction_loss + 0.05 * kl
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
with torch.no_grad():
    generated = model.decoder(torch.randn(8, 2))
assert generated.shape == (8, 2) and torch.isfinite(loss)
print(f'loss={loss.item():.4f} | reconstruction={reconstruction_loss.item():.4f} | KL={kl.item():.4f}')
print('generated sample shape:', tuple(generated.shape))